# Bucket 1 vs Bucket 2 PnL Attribution (Last Week)

This notebook performs **correct weekly attribution** using cumulative accounting snapshots:

1. select baseline snapshot (last run before window)
2. select end snapshot (latest run in window)
3. compute symbol-level deltas (`end - baseline`)
4. aggregate to underlying net PnL for `bucket_1` and `bucket_2`

It also validates that notebook results reconcile to `ibkr_accounting` `totals.json` and that split-method metadata is consistent across dates.

In [18]:
from __future__ import annotations

from datetime import timedelta
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_repo_root(start: Path) -> Path:
    """Walk upward until we find the repo root containing data/runs."""
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "runs").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/runs")


repo_root = find_repo_root(Path.cwd())
REPO_ROOT = repo_root  # alias for downstream cells that use uppercase name
runs_dir = repo_root / "data" / "runs"
etf_screened_path = repo_root / "data" / "etf_screened_today.csv"

repo_root

WindowsPath('c:/Users/werdn/Documents/Investing/ls-algo')

In [19]:
# Discover latest available run date and build a 7-day window.
run_dirs = [
    p
    for p in runs_dir.iterdir()
    if p.is_dir() and re.fullmatch(r"\d{4}-\d{2}-\d{2}", p.name)
]
run_dates = sorted(pd.to_datetime([p.name for p in run_dirs]))

if not run_dates:
    raise RuntimeError(f"No dated run directories found in {runs_dir}")

latest_date = run_dates[-1]
start_date = latest_date - timedelta(days=6)

selected_run_dirs = []
for dt in run_dates:
    if start_date <= dt <= latest_date:
        pnl_path = runs_dir / dt.strftime("%Y-%m-%d") / "accounting" / "pnl_by_symbol.csv"
        if pnl_path.exists():
            selected_run_dirs.append((dt.normalize(), pnl_path))

if not selected_run_dirs:
    raise RuntimeError("No pnl_by_symbol.csv files found in the selected 7-day window")

selected_dates = [d.strftime("%Y-%m-%d") for d, _ in selected_run_dirs]
print(f"Latest run date: {latest_date.strftime('%Y-%m-%d')}")
print(f"Window start:    {start_date.strftime('%Y-%m-%d')}")
print("Included dates:", ", ".join(selected_dates))

Latest run date: 2026-04-15
Window start:    2026-04-09
Included dates: 2026-04-09, 2026-04-10, 2026-04-11, 2026-04-14, 2026-04-15


In [20]:
# Load symbol-level accounting and keep only bucket 1 + bucket 2.
frames = []
for run_date, csv_path in selected_run_dirs:
    df = pd.read_csv(csv_path)
    df["run_date"] = run_date
    frames.append(df)

pnl = pd.concat(frames, ignore_index=True)
pnl["run_date"] = pd.to_datetime(pnl["run_date"])

pnl = pnl[pnl["bucket"].isin(["bucket_1", "bucket_2"])].copy()

component_cols = [
    "realized_pnl",
    "unrealized_pnl",
    "dividends",
    "withholding_tax",
    "pil_dividends",
    "borrow_fees",
    "short_credit_interest",
    "other_fees",
    "bond_interest",
    "total_pnl",
]

# Ensure numeric precision is retained during groupby operations.
for col in component_cols:
    pnl[col] = pd.to_numeric(pnl[col], errors="coerce").fillna(0.0)

print(f"Rows loaded (bucket 1 + 2): {len(pnl):,}")
pnl[["run_date", "bucket", "symbol", "underlying", "pair", "total_pnl"]].head(10)

Rows loaded (bucket 1 + 2): 2,405


,run_date,bucket,symbol,underlying,pair,total_pnl
0,2026-04-09,bucket_1,AAOI,AAOI,AAOI (spot),"5,977.14"
1,2026-04-09,bucket_1,MRVL,MRVL,MRVL (spot),"4,308.22"
2,2026-04-09,bucket_1,SMUP,SMR,SMR | SMUP,"3,699.87"
3,2026-04-09,bucket_1,NFLX,NFLX,NFLX (spot),"3,659.44"
4,2026-04-09,bucket_1,TER,TER,TER (spot),"3,478.32"
5,2026-04-09,bucket_1,VRT,VRT,VRT (spot),"3,398.19"
6,2026-04-09,bucket_1,MDBX,MDB,MDB | MDBX,"3,374.54"
7,2026-04-09,bucket_1,PLUG,PLUG,PLUG (spot),"3,373.03"
8,2026-04-09,bucket_1,CRMU,CRML,CRML | CRMU,"3,190.55"
9,2026-04-09,bucket_1,FIGG,FIG,FIG | FIGG,"2,955.81"


## Why prior numbers looked too large

`ibkr_accounting` outputs are **cumulative snapshots** (YTD-style components). Summing multiple run dates double-counts prior days.

For a true last-week attribution, compute **delta attribution** between:

- baseline snapshot = latest run **before** the 7-day window
- end snapshot = latest run in the 7-day window

In [21]:
# Optional: reconcile against totals.json bucket_pnl for each included date.
recon_rows = []
for ds in selected_dates:
    totals_path = runs_dir / ds / "accounting" / "totals.json"
    if not totals_path.exists():
        continue
    obj = pd.read_json(totals_path, typ="series")
    bucket_pnl = obj.get("bucket_pnl", {})
    recon_rows.append(
        {
            "run_date": ds,
            "bucket_1_total_from_totals_json": float(bucket_pnl.get("bucket_1", 0.0)),
            "bucket_2_total_from_totals_json": float(bucket_pnl.get("bucket_2", 0.0)),
        }
    )

recon = pd.DataFrame(recon_rows)
recon

,run_date,bucket_1_total_from_totals_json,bucket_2_total_from_totals_json
0,2026-04-09,"11,701.41","4,646.69"
1,2026-04-10,"9,313.08","6,415.36"
2,2026-04-11,"8,421.27","8,301.31"
3,2026-04-14,"7,729.28","8,907.90"
4,2026-04-15,"4,695.27","9,087.49"


## Weekly insight summary (delta basis)

This section summarizes the main PnL drivers over the selected window using snapshot deltas (`end - baseline`).

## Alternative attribution: hedge-position weighted (non-circular)

This section rebuilds bucket 1/2 attribution without using `total_pnl`-weighted spot splits.

Method:

- ETF symbols are assigned by beta (`>1.5 -> bucket_1`, `0<beta<=1.5 -> bucket_2`)
- non-ETF symbols are split by each underlying's hedge mix using **beta-adjusted absolute ETF position notional**
  measured from baseline and end snapshots (averaged)
- if an underlying has no measured hedge mix, fallback to universe beta mix

This keeps attribution tied to hedge structure rather than same-day PnL magnitude.

In [22]:
# Build hedge-position-weighted bucket attribution for b1/b2
import sys
from pathlib import Path

try:
    _repo_for_import = REPO_ROOT
except NameError:
    _p = Path.cwd().resolve()
    while _p != _p.parent and not (_p / "ibkr_accounting.py").exists():
        _p = _p.parent
    _repo_for_import = _p

if str(_repo_for_import) not in sys.path:
    sys.path.insert(0, str(_repo_for_import))

from ibkr_accounting import (
    SUPPLEMENTAL_ETF_MAP,
    load_etf_beta_map,
    load_etf_to_under_map,
    parse_open_positions,
)

import re
import pandas as pd

if "runs_dir" not in globals():
    runs_dir = _repo_for_import / "data" / "runs"
if "etf_screened_path" not in globals():
    etf_screened_path = _repo_for_import / "data" / "etf_screened_today.csv"

if "baseline_date" not in globals() or "end_date" not in globals():
    _all_dates = sorted(
        pd.to_datetime([
            p.name
            for p in runs_dir.iterdir()
            if p.is_dir() and re.fullmatch(r"\d{4}-\d{2}-\d{2}", p.name)
        ])
    )
    if not _all_dates:
        raise RuntimeError(f"No run dates found under {runs_dir}")
    end_date = _all_dates[-1]
    baseline_date = _all_dates[-2] if len(_all_dates) >= 2 else _all_dates[-1]

if "delta" not in globals() or "component_cols" not in globals():
    # Self-heal: rebuild snapshot delta locally if upstream cells were removed.
    _base_p = runs_dir / baseline_date.strftime("%Y-%m-%d") / "accounting" / "pnl_by_symbol.csv"
    _end_p = runs_dir / end_date.strftime("%Y-%m-%d") / "accounting" / "pnl_by_symbol.csv"
    if not _base_p.exists() or not _end_p.exists():
        raise RuntimeError(
            "Could not rebuild `delta`: missing pnl_by_symbol.csv for baseline/end. "
            f"baseline={_base_p.exists()} end={_end_p.exists()}"
        )

    _base = pd.read_csv(_base_p)
    _end = pd.read_csv(_end_p)

    _candidate_components = [
        "realized_pnl",
        "unrealized_pnl",
        "dividend_pnl",
        "borrow_fee_pnl",
        "interest_pnl",
        "other_pnl",
        "tax_pnl",
        "fx_pnl",
        "commissions_pnl",
        "total_pnl",
    ]
    component_cols = [c for c in _candidate_components if c in _base.columns and c in _end.columns]
    if not component_cols:
        raise RuntimeError(
            "Could not infer PnL component columns while rebuilding `delta`. "
            f"base_cols={list(_base.columns)}"
        )

    _meta_cols = [c for c in ["symbol", "underlying", "pair", "bucket", "description"] if c in _base.columns and c in _end.columns]
    if "symbol" not in _meta_cols:
        raise RuntimeError("Could not rebuild `delta`: `symbol` column missing in pnl_by_symbol.csv")

    _base2 = _base[_meta_cols + component_cols].copy()
    _end2 = _end[_meta_cols + component_cols].copy()
    for _c in component_cols:
        _base2[_c] = pd.to_numeric(_base2[_c], errors="coerce").fillna(0.0)
        _end2[_c] = pd.to_numeric(_end2[_c], errors="coerce").fillna(0.0)

    _agg_keys = [c for c in _meta_cols if c != "description"]
    _base_agg = _base2.groupby(_agg_keys, as_index=False)[component_cols].sum()
    _end_agg = _end2.groupby(_agg_keys, as_index=False)[component_cols].sum()

    _m = _end_agg.merge(_base_agg, on=_agg_keys, how="outer", suffixes=("_end", "_base")).fillna(0.0)
    for _c in component_cols:
        _m[f"{_c}_delta"] = pd.to_numeric(_m[f"{_c}_end"], errors="coerce").fillna(0.0) - pd.to_numeric(_m[f"{_c}_base"], errors="coerce").fillna(0.0)

    _desc = _end2[["symbol"] + (["description"] if "description" in _end2.columns else [])].drop_duplicates("symbol")
    delta = _m.merge(_desc, on="symbol", how="left") if "description" in _desc.columns else _m

_, etf_to_beta_map_alt = load_etf_beta_map(etf_screened_path)
etf_to_under_map_alt = load_etf_to_under_map(etf_screened_path)
etf_to_under_all_alt = {**etf_to_under_map_alt, **SUPPLEMENTAL_ETF_MAP}


def _bucket_for_etf_beta(beta: float | None) -> str | None:
    if beta is None:
        return None
    if beta > 1.5:
        return "bucket_1"
    if 0 < beta <= 1.5:
        return "bucket_2"
    return None


# Universe fallback (same philosophy as ibkr_accounting fallback map)
_b1_u = 0.0
_b2_u = 0.0
for _sym, _beta in etf_to_beta_map_alt.items():
    _bkt = _bucket_for_etf_beta(_beta)
    if _bkt is None:
        continue
    _w = abs(float(_beta))
    if _bkt == "bucket_1":
        _b1_u += _w
    else:
        _b2_u += _w

_univ_total = _b1_u + _b2_u
if _univ_total > 0:
    universe_ratio = {"b1": _b1_u / _univ_total, "b2": _b2_u / _univ_total}
else:
    universe_ratio = {"b1": 0.5, "b2": 0.5}


def _position_ratio_map_for_date(d: pd.Timestamp) -> dict[str, dict[str, float]]:
    p = runs_dir / d.strftime("%Y-%m-%d") / "ibkr_flex" / "flex_positions.xml"
    if not p.exists():
        return {}

    pos = parse_open_positions(p)
    if pos.empty:
        return {}

    tmp = pos.copy()
    tmp["symbol"] = tmp["symbol"].astype(str).str.upper().str.strip()
    tmp["beta"] = tmp["symbol"].map(etf_to_beta_map_alt)
    tmp["bucket"] = tmp["beta"].map(_bucket_for_etf_beta)
    tmp = tmp[tmp["bucket"].isin(["bucket_1", "bucket_2"])].copy()
    if tmp.empty:
        return {}

    tmp["underlying"] = tmp["symbol"].map(etf_to_under_all_alt).fillna(tmp["symbol"])

    # Use the best available notional column from parse_open_positions output.
    # Current ibkr_accounting emits `positionValue_base`, but we keep legacy fallbacks
    # for notebook portability across historical environments.
    _notional_col = next(
        (
            c
            for c in ["positionValue_base", "positionValueBase", "positionValue", "positionValueSOD"]
            if c in tmp.columns
        ),
        None,
    )
    if _notional_col is None:
        raise KeyError(
            "No position notional column found in open positions. "
            f"Available columns: {list(tmp.columns)}"
        )

    tmp["hedge_notional"] = (
        pd.to_numeric(tmp[_notional_col], errors="coerce").fillna(0.0).abs()
        * pd.to_numeric(tmp["beta"], errors="coerce").fillna(0.0).abs()
    )

    grp = (
        tmp.groupby(["underlying", "bucket"], as_index=False)["hedge_notional"]
        .sum()
    )
    if grp.empty:
        return {}

    piv = grp.pivot(index="underlying", columns="bucket", values="hedge_notional").fillna(0.0)
    out = {}
    for u, row in piv.iterrows():
        b1 = float(row.get("bucket_1", 0.0))
        b2 = float(row.get("bucket_2", 0.0))
        s = b1 + b2
        if s > 0:
            out[str(u)] = {"b1": b1 / s, "b2": b2 / s}
    return out


baseline_ratio_map = _position_ratio_map_for_date(baseline_date)
end_ratio_map = _position_ratio_map_for_date(end_date)

# Combine baseline+end hedge structure; if one side missing, use available side
combined_ratio_map: dict[str, dict[str, float]] = {}
all_underlyings_ratio = set(baseline_ratio_map) | set(end_ratio_map)
for _u in all_underlyings_ratio:
    _b1_vals = []
    _b2_vals = []
    if _u in baseline_ratio_map:
        _b1_vals.append(float(baseline_ratio_map[_u]["b1"]))
        _b2_vals.append(float(baseline_ratio_map[_u]["b2"]))
    if _u in end_ratio_map:
        _b1_vals.append(float(end_ratio_map[_u]["b1"]))
        _b2_vals.append(float(end_ratio_map[_u]["b2"]))
    if _b1_vals:
        _b1 = float(sum(_b1_vals) / len(_b1_vals))
        _b2 = float(sum(_b2_vals) / len(_b2_vals))
        _s = _b1 + _b2
        if _s > 0:
            combined_ratio_map[_u] = {"b1": _b1 / _s, "b2": _b2 / _s}


# Rebuild symbol total deltas from already-computed delta table (unsplit at symbol level first)
# Use columns that actually exist in `delta` to stay robust across schema versions.
component_delta_cols = sorted(
    [
        c
        for c in delta.columns
        if c.endswith("_delta") and c != "total_pnl_delta"
    ]
)
metric_cols = component_delta_cols + (["total_pnl_delta"] if "total_pnl_delta" in delta.columns else [])
if not metric_cols:
    raise RuntimeError(
        "No delta metric columns found in `delta`. "
        f"Available columns: {list(delta.columns)}"
    )

symbol_meta = (
    delta.sort_values("symbol")
    .groupby("symbol", as_index=False)
    .agg(
        underlying=("underlying", "first"),
        pair=("pair", "first"),
        description=("description", "first"),
    )
)

symbol_delta_total = (
    delta.groupby("symbol", as_index=False)[metric_cols]
    .sum()
    .merge(symbol_meta, on="symbol", how="left")
)


def _num_scalar(v) -> float:
    if isinstance(v, pd.Series):
        return float(pd.to_numeric(v, errors="coerce").fillna(0.0).sum())
    return float(pd.to_numeric(v, errors="coerce"))


def _ratio_for_underlying(u: str) -> dict[str, float]:
    if u in combined_ratio_map:
        return combined_ratio_map[u]
    return universe_ratio


alt_rows = []
for _, r in symbol_delta_total.iterrows():
    sym = str(r["symbol"])
    und = str(r.get("underlying", "") or sym)
    beta = etf_to_beta_map_alt.get(sym)
    bkt = _bucket_for_etf_beta(beta)
    is_known_etf = sym in etf_to_beta_map_alt

    if bkt in {"bucket_1", "bucket_2"}:
        out = {k: _num_scalar(r[k]) for k in metric_cols}
        out.update({
            "symbol": sym,
            "underlying": und,
            "bucket": bkt,
            "split_ratio": 1.0,
            "split_source": "etf_beta",
        })
        alt_rows.append(out)
        continue

    if is_known_etf and beta is not None and beta < 0:
        # Inverse ETFs belong to bucket_3; exclude them from b1/b2 alternative split.
        out = {k: _num_scalar(r[k]) for k in metric_cols}
        out.update({
            "symbol": sym,
            "underlying": und,
            "bucket": "bucket_3",
            "split_ratio": 1.0,
            "split_source": "etf_beta_inverse",
        })
        alt_rows.append(out)
        continue

    # non-ETF -> split by hedge-position ratio (only b1/b2)
    ratio = _ratio_for_underlying(und)
    for bkt_label, rk in [("bucket_1", "b1"), ("bucket_2", "b2")]:
        rr = float(ratio[rk])
        out = {k: _num_scalar(r[k]) * rr for k in metric_cols}
        out.update({
            "symbol": sym,
            "underlying": und,
            "bucket": bkt_label,
            "split_ratio": rr,
            "split_source": "hedge_position_ratio" if und in combined_ratio_map else "universe_fallback",
        })
        alt_rows.append(out)

alt_delta = pd.DataFrame(alt_rows)

alt_underlying_b12 = (
    alt_delta.groupby(["underlying", "bucket"], as_index=False)["total_pnl_delta"]
    .sum()
    .pivot(index="underlying", columns="bucket", values="total_pnl_delta")
    .fillna(0.0)
    .reset_index()
)
for c in ["bucket_1", "bucket_2"]:
    if c not in alt_underlying_b12.columns:
        alt_underlying_b12[c] = 0.0
alt_underlying_b12["total_b12_delta"] = alt_underlying_b12["bucket_1"] + alt_underlying_b12["bucket_2"]
alt_underlying_b12 = alt_underlying_b12.sort_values("total_b12_delta")

print(f"Combined hedge ratio underlyings: {len(combined_ratio_map)}")
print("Universe fallback ratio:", universe_ratio)
print("Top 15 most negative underlyings (alt attribution):")
display(alt_underlying_b12.head(15))
print("Top 15 most positive underlyings (alt attribution):")
display(alt_underlying_b12.tail(15).sort_values("total_b12_delta", ascending=False))

KeyError: "Columns not found: 'short_credit_interest_delta', 'withholding_tax_delta', 'bond_interest_delta', 'borrow_fees_delta', 'pil_dividends_delta', 'other_fees_delta', 'dividends_delta'"

In [ ]:
# Reconciliation checks for the alternative method
import json
# 1) b1+b2 combined must equal original b1+b2 combined delta (only split method changed)
orig_b12_combined = float(delta.loc[delta["bucket"].isin(["bucket_1", "bucket_2"]), "total_pnl_delta"].sum())
alt_b12_combined = float(alt_delta.loc[alt_delta["bucket"].isin(["bucket_1", "bucket_2"]), "total_pnl_delta"].sum())
combined_diff = alt_b12_combined - orig_b12_combined

# 2) Compare bucket deltas vs current ibkr_accounting bucket outputs
alt_b1 = float(alt_delta.loc[alt_delta["bucket"] == "bucket_1", "total_pnl_delta"].sum())
alt_b2 = float(alt_delta.loc[alt_delta["bucket"] == "bucket_2", "total_pnl_delta"].sum())

base_totals = json.loads((runs_dir / baseline_date.strftime("%Y-%m-%d") / "accounting" / "totals.json").read_text(encoding="utf-8"))
end_totals = json.loads((runs_dir / end_date.strftime("%Y-%m-%d") / "accounting" / "totals.json").read_text(encoding="utf-8"))

ibkr_b1 = float(end_totals.get("bucket_pnl", {}).get("bucket_1", 0.0) - base_totals.get("bucket_pnl", {}).get("bucket_1", 0.0))
ibkr_b2 = float(end_totals.get("bucket_pnl", {}).get("bucket_2", 0.0) - base_totals.get("bucket_pnl", {}).get("bucket_2", 0.0))

recon_alt = pd.DataFrame([
    {
        "metric": "b1_delta",
        "alternative_hedge_weighted": alt_b1,
        "current_ibkr_accounting": ibkr_b1,
        "difference_alt_minus_current": alt_b1 - ibkr_b1,
    },
    {
        "metric": "b2_delta",
        "alternative_hedge_weighted": alt_b2,
        "current_ibkr_accounting": ibkr_b2,
        "difference_alt_minus_current": alt_b2 - ibkr_b2,
    },
    {
        "metric": "b12_combined_delta",
        "alternative_hedge_weighted": alt_b12_combined,
        "current_ibkr_accounting": orig_b12_combined,
        "difference_alt_minus_current": combined_diff,
    },
])

display(recon_alt)

# Symbol-level conservation diagnostic: alt b1+b2 should match original b1+b2 exactly per symbol.
orig_sym_b12 = (
    delta.loc[delta["bucket"].isin(["bucket_1", "bucket_2"])]
    .groupby("symbol", as_index=False)["total_pnl_delta"]
    .sum()
    .rename(columns={"total_pnl_delta": "orig_b12"})
)
alt_sym_b12 = (
    alt_delta.loc[alt_delta["bucket"].isin(["bucket_1", "bucket_2"])]
    .groupby("symbol", as_index=False)["total_pnl_delta"]
    .sum()
    .rename(columns={"total_pnl_delta": "alt_b12"})
)
sym_recon = orig_sym_b12.merge(alt_sym_b12, on="symbol", how="outer").fillna(0.0)
sym_recon["diff"] = sym_recon["alt_b12"] - sym_recon["orig_b12"]
max_sym_abs = float(sym_recon["diff"].abs().max()) if not sym_recon.empty else 0.0

# Use a practical tolerance to avoid false breaks from floating aggregation noise.
_tol = 1e-4
if abs(combined_diff) >= _tol:
    print("Top 20 symbol-level conservation differences:")
    display(sym_recon.sort_values("diff", key=lambda s: s.abs(), ascending=False).head(20))
    raise AssertionError(
        "Alternative attribution failed conservation check for b1+b2 combined delta.\n"
        + recon_alt.to_string(index=False)
        + f"\nmax_symbol_abs_diff={max_sym_abs:.6f}"
    )

# Netting diagnostics (why b1 and b2 can nearly cancel)
gvn = alt_underlying_b12.copy()
gvn["gross_pair"] = gvn["bucket_1"].abs() + gvn["bucket_2"].abs()
gvn["offset_amt"] = gvn["gross_pair"] - gvn["total_b12_delta"].abs()
gvn["offset_pct"] = np.where(gvn["gross_pair"] > 0, gvn["offset_amt"] / gvn["gross_pair"], 0.0)

print("Top 15 underlyings by offset_pct (most netting):")
display(gvn.sort_values("offset_pct", ascending=False).head(15))

print(f"Conservation check OK. Combined b1+b2 delta unchanged: {alt_b12_combined:,.2f}")

,metric,alternative_hedge_weighted,current_ibkr_accounting,difference_alt_minus_current
0,b1_delta,881.99,"-3,034.01","3,916.00"
1,b2_delta,-978.60,179.59,"-1,158.20"
2,b12_combined_delta,-96.62,"-2,854.42","2,757.80"


AssertionError: Alternative attribution failed conservation check for b1+b2 combined delta.
            metric  alternative_hedge_weighted  current_ibkr_accounting  difference_alt_minus_current
          b1_delta                      881.99                -3,034.01                      3,916.00
          b2_delta                     -978.60                   179.59                     -1,158.20
b12_combined_delta                      -96.62                -2,854.42                      2,757.80